In [16]:
import os
import pandas as pd
import requests
import json
from dotenv import load_dotenv

In [23]:
ENDPOINT = "https://acleddata.com/api/acled/read?_format=json"
start_date = "2025-01-01"
end_date = "2025-12-31"
countries = ["Sudan"]

load_dotenv()
username=os.getenv("ACLED_USERNAME")
password=os.getenv("ACLED_PASSWORD")
token_url=os.getenv("ACLED_TOKEN_URL")

In [19]:
def get_access_token():
    headers = {
    "Content-Type": "application/x-www-form-urlencoded",
    }
    data = {
        "username": username,
        "password": password,
        "grant_type": "password",
        "client_id": "acled",
    }

    # Get response from url
    response = requests.post(
        token_url, headers=headers, data=data, timeout=20
    )

    # Return access token if successful.
    if response.status_code == 200:  # noqa: PLR2004
        token_data = response.json()
        return token_data["access_token"]
    raise Exception(
            f"Failed to get access token: {response.status_code} {response.text}"
        )

In [24]:
def build_params(countries, start_date, end_date, sub_event_type) -> str:
    """Build the API parameters dictionary."""
    params = {
        "country": "|".join(countries),  # type: ignore
        "event_date": f"{start_date}|{end_date}",
        "event_date_where": "BETWEEN",
        "sub_event_type": "|".join(sub_event_type),
    }
    return params

In [25]:
def get_data(countries, start_date, end_date, sub_event_type):
    params = build_params(countries, start_date, end_date, sub_event_type)
    access_token = get_access_token()

    params["page"] = 1
    request_end = False
    r_dfs = []

    while not request_end:
        r = requests.get(
            url=ENDPOINT,
            headers={
                "Authorization": f"Bearer {access_token}",
                "Content-Type": "application/json",
            },
            params=params,
            timeout=20, # Remove hard coding
        )
        if r.ok:
            r_df = pd.DataFrame.from_dict(r.json()["data"])
            r_dfs.append(r_df)

            # Check if last request (return size is less than the API limit),
            # otherwise increment page for the next paginated response
            if len(r.json()["data"]) < 5000:
                request_end = True
            else:
                params["page"] += 1
        else:
            raise requests.RequestException(
                f"HTTP Code: {r.status_code}, Status: {r.reason}"
            )
    final_df = pd.concat(r_dfs)
    return final_df


In [26]:
test = get_data(countries, start_date, end_date, "Battles")

In [27]:
test

""


In [30]:

params = build_params(countries, start_date, end_date, "Battles")
access_token = get_access_token()

params["page"] = 1

r = requests.get(
    url=ENDPOINT,
    headers={
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    },
    params=params,
    timeout=20, # TODO: Remove hard coding
)
print(r.status_code)


200
